# Module 4: Fashion Image Preprocessing
## Image Standardization, Aspect-Ratio Letterboxing, Normalization & tf.data Pipelines

This notebook demonstrates:
1. Aspect-ratio preserving resizing (Letterbox padding vs Direct stretching).
2. Multi-protocol pixel normalization ($[0, 1]$, $[-1, 1]$, ImageNet mean/std).
3. Real-time data augmentation for training robustness.
4. Memory-efficient asynchronous batch streaming using `tf.data.Dataset`.

In [ ]:
import sys
from pathlib import Path

# Ensure project root in sys.path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf

from src.preprocessing import FashionImagePreprocessor, create_tf_dataset, get_data_augmentation_pipeline
from src.config import SPLITS_CSV, TARGET_IMG_SIZE

plt.rcParams["figure.figsize"] = (12, 6)

### 1. Load Dataset Splits & Initialize Preprocessor

In [ ]:
df_splits = pd.read_csv(SPLITS_CSV)
print(f"Total items: {len(df_splits):,}")
print("Split distribution:")
print(df_splits["split"].value_counts())

preprocessor = FashionImagePreprocessor(target_size=(224, 224))
sample_row = df_splits[df_splits["canonical_category"].isin(["Dress", "Jeans", "Shoes"])].iloc[0]
sample_path = sample_row["image_path"]
print(f"\nSample item: {sample_row['productDisplayName']} ({sample_row['canonical_category']})")

### 2. Aspect-Ratio Letterbox Padding vs Direct Stretching
In fashion recommendation, preserving garment silhouettes is critical (e.g. dresses and jeans shouldn't become squished or widened).

In [ ]:
raw_img = preprocessor.load_image(sample_path)
direct_img = preprocessor.direct_resize(raw_img, target_size=(224, 224))
letterbox_img = preprocessor.resize_and_pad(raw_img, target_size=(224, 224))

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(raw_img)
axes[0].set_title(f"Original Image\nShape: {raw_img.size} (W x H)", fontweight="bold")
axes[0].axis("off")

axes[1].imshow(direct_img)
axes[1].set_title("Direct Resize (Distorted)\nShape: 224 x 224", fontweight="bold")
axes[1].axis("off")

axes[2].imshow(letterbox_img)
axes[2].set_title("Letterbox Padded (Preserved)\nShape: 224 x 224", fontweight="bold")
axes[2].axis("off")

plt.tight_layout()
plt.show()

### 3. Normalization Protocols Comparison
Comparing standard $[0, 1]$ scaling against ImageNet standardization.

In [ ]:
img_arr = np.array(letterbox_img)
norm_0_1 = preprocessor.normalize(img_arr, mode="scale_0_1")
norm_m1_1 = preprocessor.normalize(img_arr, mode="tf_minus1_to_1")
norm_imagenet = preprocessor.normalize(img_arr, mode="imagenet")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].hist(norm_0_1.ravel(), bins=50, color="skyblue", edgecolor="black")
axes[0].set_title(f"Scale [0, 1]\nMin: {norm_0_1.min():.2f}, Max: {norm_0_1.max():.2f}", fontweight="bold")

axes[1].hist(norm_m1_1.ravel(), bins=50, color="salmon", edgecolor="black")
axes[1].set_title(f"Scale [-1, 1]\nMin: {norm_m1_1.min():.2f}, Max: {norm_m1_1.max():.2f}", fontweight="bold")

axes[2].hist(norm_imagenet.ravel(), bins=50, color="lightgreen", edgecolor="black")
axes[2].set_title(f"ImageNet Standardized\nMean: {norm_imagenet.mean():.2f}, Std: {norm_imagenet.std():.2f}", fontweight="bold")

plt.tight_layout()
plt.show()

### 4. Training Data Augmentation Pipeline
Simulating random flips, subtle rotations, and zoom perturbations to build robust model representations.

In [ ]:
aug_pipeline = get_data_augmentation_pipeline()
img_tensor = tf.expand_dims(tf.constant(norm_0_1), axis=0)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
axes[0].imshow(norm_0_1)
axes[0].set_title("Original Preprocessed", fontweight="bold")
axes[0].axis("off")

for i in range(1, 5):
    augmented = aug_pipeline(img_tensor, training=True)[0].numpy()
    axes[i].imshow(np.clip(augmented, 0, 1))
    axes[i].set_title(f"Augmented Variant #{i}", fontweight="bold")
    axes[i].axis("off")

plt.tight_layout()
plt.show()

### 5. Memory-Efficient Asynchronous `tf.data` Batch Streaming

In [ ]:
train_df = df_splits[df_splits["split"] == "train"].head(100)
train_ds = create_tf_dataset(train_df, batch_size=16, is_training=True)

for batch_images, batch_labels in train_ds.take(1):
    print(f"Streamed Batch Images Shape : {batch_images.shape} (Batch, H, W, Channels)")
    print(f"Streamed Batch Labels Shape : {batch_labels.shape} (Batch, Num_Classes)")
    print(f"Tensor dtype                : {batch_images.dtype}")
    print(f"Pixel Range                 : [{tf.reduce_min(batch_images):.3f}, {tf.reduce_max(batch_images):.3f}]")
    break